# 05 · Monte Carlo on Kaggle (harder experiment, detector v2)

Settings (right sidebar): **Accelerator = GPU T4 x2** (the P100 is too old for Blender 5), **Internet = On** (needs a phone-verified account), and add the private dataset that holds `best.pt` (the v2 detector checkpoint).

A 20-landing pilot (`SEEDS = (0, 20)`, ~12 min) checks that everything works. For the real run set `SEEDS` to a big range and use **Save Version → Save & Run All (Commit)** so it runs without a browser tab. The runner stops itself before the session limit, and everything in the output folder is kept.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import os
# the v2 detector was uploaded as two parts (the upload limit was 10 MB per file); join them back
!find /kaggle/input -name "best.pt.part*" | sort | xargs cat > /tmp/best.pt
CKPT = '/tmp/best.pt'
assert os.path.getsize(CKPT) == 17520863, os.path.getsize(CKPT)
SEEDS = (0, 1000)        # all landings; seeds that already have a JSON file in OUT are skipped (resumable). Pilot: (0, 20)
OUT = '/kaggle/working/mc_random_safe'

In [ ]:
# bring in landings from earlier sessions (add that notebook's output as an input); they are skipped, not re-flown
!mkdir -p {OUT}
!find /kaggle/input -path "*mc_random_safe*" -name "landing_*.json" -exec cp -n {{}} {OUT}/ \;
!ls {OUT} | wc -l
# prints one line per minute; stops itself at 8.5 h and then writes summary.json and sensitivity.json into OUT
!python /tmp/touch_down/scripts/kaggle_run.py --ckpt {CKPT} --seeds {SEEDS[0]} {SEEDS[1]} --out {OUT}